# 1. Information about the submission

## 1.1 Name and number of the assignment

Assignment 2: Semantic Role Labelling

## 1.2 Student name

Egor Serov

## 1.3 Codalab user ID / nickname / username

iwanttodie

## 1.4 Additional comments

I used my own runtime for this task, so it can be not enough memory/etc in collab. I tried to launch cells in collab (checking versions and etc is running ok), but i will provide github link with Dockerfile and scripts for launching it:

https://github.com/awesomeslayer/IntroToNLP2025

# 2. Technical Report

*Use Section 2 to describe results of your experiments as you would do writing a paper about your results. DO NOT insert code in this part. Only insert plots and tables summarizing results as needed. Use formulas if needed do described your methodology. The code is provided in Section 3.*

## 2.1 Methodology

***Enter here** a detailed description of the method used in your submission(s) to Codalab. The description should be at least 2-4 paragraphs featuring the following: type of the model, meta-parameters, how did you select meta-parameters, any further modifications of the out-of-the-box solutions, etc. The text is markdown and you can use math environment to write formulas:*

$\hat{y}=\beta_0 + \sum^p_{j=1} x_j \beta_j$

Also you can insert images as needed:

![image](https://upload.wikimedia.org/wikipedia/commons/6/6d/Exam_pass_logistic_curve.jpeg)

*This part of the should contain description of all methods that you tried and, most importantly, that worked the best for you. Here you can include some tricks of your preprocessing, description of the models and motivation of their usage, the description of the training process details (train-test split, cross-validation, etc.). So, everything valuable that will help us to understand the scope of your work and reproduce your pipeline*

*The 'innovativeness' scores will be assigned based on the content of this part. In case you just use a 'drop-in' baseline model these scores will be low. In case you made some clever modification of the model which improves the result this score will be low. However, it does not make sense do describe some modifications which are creative but does not work at all. Try different approaches, models, play with preprocessing, hyperparameters, etc. It is OK to reimplement some already existing approach. It is OK that some of your experiments did not work as you expected. Show us that you used your creativity and ran several experiments.*

## 2.2 Discussion of results

***Enter here** a discussion of results and a summary of the experiment. Here we want to see the final table with comparison of the baseline and all tried approaches you decided to report. Even if some method did not bring you to the top of the leaderboard, you should nevertheless indicate this result and a discussion, why, in your opinion, some approach worked and another failed. Interesting findings in the discussion will be a plus.*

Method | Precision | Recall
--- | --- | ---
Baseline | 0.88 | 0.77
My great method 1 | 0.99 | 0.11
My great method 2 | 0.90 | 0.90

*If relevant insert plots and historgams in this section e.g. testing variation of the score with respect to some parameters e.g. learning rate or size of the input dataset, etc. Please do not use code to generate plots, instead just insert images as shown below. Plots could be generated from code in Section 3. *

![image](https://upload.wikimedia.org/wikipedia/commons/thumb/d/d2/Sine_and_Cosine.svg/640px-Sine_and_Cosine.svg.png)

# 3. Code

*Enter here all code used to produce your results submitted to Codalab. Add some comments and subsections to navigate though your solution.*

*In this part you are expected to develop yourself a solution of the task and provide a reproducible code:*
- *Using Python 3;*
- *Contains code for installation of all dependencies;*
- *Contains code for downloading of all the datasets used*;
- *Contains the code for reproducing your results (in other words, if a tester downloads your notebook she should be able to run cell-by-cell the code and obtain your experimental results as described in the methodology section)*.


*As a result, you code will be graded according to these criteria:*
- ***Readability**: your code should be well-structured preferably with indicated parts of your approach (Preprocessing, Model training, Evaluation, etc.).*
- ***Reproducibility**: your code should be reproduced without any mistakes with “Run all” mode (obtaining experimental part).*


## 3.1 Requirements

In [1]:
import os
import zipfile

os.environ["TOKENIZERS_PARALLELISM"] = "false"

!mkdir -p plots tables

!pip install --upgrade pip
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install transformers datasets seqeval accelerate scikit-learn pandas tqdm tensorboard matplotlib seaborn

Looking in indexes: https://download.pytorch.org/whl/cu118


## 3.2 Download the data

In [2]:
import os

if not os.path.exists("semantic-role-labelling"):
    !git clone https://github.com/s-nlp/semantic-role-labelling

print("Data downloaded.")

Data downloaded.


In [3]:
import os
import zipfile
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from datasets import Dataset as HFDataset, DatasetDict
import evaluate
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer,
    DataCollatorForTokenClassification,
    set_seed
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)
sns.set_theme(style="whitegrid")

for d in ["plots", "tables", "submissions", "models"]:
    os.makedirs(d, exist_ok=True)

## 3.3 Preprocessing

In [4]:
def read_dataset(filename, splitter="\t"):
    data = []
    sentence = []
    tags = []
    with open(filename, encoding='utf-8') as f:
        for line in f:
            if not line.isspace():
                parts = line.split(splitter)
                word = parts[0]
                tag = parts[1].strip() if len(parts) > 1 else "O"
                sentence.append(word)
                tags.append(tag)
            else:
                if len(sentence) > 0:
                    data.append({"tokens": sentence, "ner_tags": tags})
                sentence = []
                tags = []
    if len(sentence) > 0:
        data.append({"tokens": sentence, "ner_tags": tags})
    return data

raw_train_data = read_dataset("semantic-role-labelling/train.tsv")
raw_dev_data = read_dataset("semantic-role-labelling/dev_no_answers.tsv", splitter="\n")
raw_test_data = read_dataset("semantic-role-labelling/test_no_answers.tsv", splitter="\n")

print(f"Train samples: {len(raw_train_data)}")
print(f"Dev samples:   {len(raw_dev_data)}")
print(f"Test samples:  {len(raw_test_data)}")

Train samples: 2334
Dev samples:   283
Test samples:  360


In [5]:
label_list = ["O", "B-Object", "I-Object", "B-Aspect", "I-Aspect", "B-Predicate", "I-Predicate"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

def convert_to_ids(data):
    processed = []
    for item in data:
        processed.append({
            "tokens": item["tokens"],
            "ner_tags": [label2id.get(t, 0) for t in item["ner_tags"]]
        })
    return processed

train_processed = convert_to_ids(raw_train_data)
train_subset, val_subset = train_test_split(train_processed, test_size=0.1, random_state=42)

MODEL_CHECKPOINT = "roberta-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, add_prefix_space=True)

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    
    if "ner_tags" in examples:
        labels = []
        for i, label in enumerate(examples["ner_tags"]):
            word_ids = tokenized_inputs.word_ids(batch_index=i)
            previous_word_idx = None
            label_ids = []
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)
                elif word_idx != previous_word_idx:
                    label_ids.append(label[word_idx])
                else:
                    label_ids.append(-100)
                previous_word_idx = word_idx
            labels.append(label_ids)
        tokenized_inputs["labels"] = labels
    return tokenized_inputs

dataset = DatasetDict({
    "train": HFDataset.from_list(train_subset),
    "validation": HFDataset.from_list(val_subset),
    "dev": HFDataset.from_list([{"tokens": x["tokens"]} for x in raw_dev_data]),
    "test": HFDataset.from_list([{"tokens": x["tokens"]} for x in raw_test_data])
})

tokenized_datasets = dataset.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/2100 [00:00<?, ? examples/s]

Map:   0%|          | 0/234 [00:00<?, ? examples/s]

Map:   0%|          | 0/283 [00:00<?, ? examples/s]

Map:   0%|          | 0/360 [00:00<?, ? examples/s]

In [6]:
word_to_ix = {}
for item in raw_train_data:
    for word in item['tokens']:
        if word not in word_to_ix:
            word_to_ix[word] = len(word_to_ix)
word_to_ix["<UNK>"] = len(word_to_ix)

def prepare_sequence(seq, to_ix):
    idxs = [to_ix.get(w, to_ix["<UNK>"]) for w in seq]
    return torch.tensor(idxs, dtype=torch.long)

class LSTMTagger(nn.Module):
    def __init__(self, embedding_dim, hidden_dim, vocab_size, tagset_size):
        super(LSTMTagger, self).__init__()
        self.word_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim)
        self.hidden2tag = nn.Linear(hidden_dim, tagset_size)

    def forward(self, sentence):
        embeds = self.word_embeddings(sentence)
        lstm_out, _ = self.lstm(embeds.view(len(sentence), 1, -1))
        tag_space = self.hidden2tag(lstm_out.view(len(sentence), -1))
        return F.log_softmax(tag_space, dim=1)

baseline_model = LSTMTagger(32, 64, len(word_to_ix), len(label2id)).to(device)
optimizer = optim.SGD(baseline_model.parameters(), lr=0.1)
loss_fn = nn.NLLLoss()

In [ ]:
print("Training Baseline LSTM...")
baseline_losses = []
baseline_model.train()
EPOCHS_LSTM = 20

for epoch in range(EPOCHS_LSTM):
    epoch_loss = 0
    for sentence_dict in tqdm(train_subset, desc=f"Baseline Epoch {epoch+1}", leave=False):
        baseline_model.zero_grad()
        sentence_in = prepare_sequence(sentence_dict['tokens'], word_to_ix).to(device)
        targets = torch.tensor(sentence_dict['ner_tags'], dtype=torch.long).to(device)
        loss = loss_fn(baseline_model(sentence_in), targets)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    baseline_losses.append(epoch_loss / len(train_subset))

Training Baseline LSTM...


Baseline Epoch 1:   0%|          | 0/2100 [00:00<?, ?it/s]

Baseline Epoch 2:   0%|          | 0/2100 [00:00<?, ?it/s]

Baseline Epoch 3:   0%|          | 0/2100 [00:00<?, ?it/s]

Baseline Epoch 4:   0%|          | 0/2100 [00:00<?, ?it/s]

...

Baseline Epoch 15:   0%|          | 0/2100 [00:00<?, ?it/s]

Baseline Epoch 16:   0%|          | 0/2100 [00:00<?, ?it/s]

Baseline Epoch 17:   0%|          | 0/2100 [00:00<?, ?it/s]

Baseline Epoch 18:   0%|          | 0/2100 [00:00<?, ?it/s]

Baseline Epoch 19:   0%|          | 0/2100 [00:00<?, ?it/s]

In [ ]:
baseline_model.eval()
seqeval = evaluate.load("seqeval")
preds_list, labels_list = [], []

with torch.no_grad():
    for sentence_dict in val_subset:
        sentence_in = prepare_sequence(sentence_dict['tokens'], word_to_ix).to(device)
        tag_scores = baseline_model(sentence_in)
        preds = tag_scores.argmax(dim=1).cpu().numpy()
        labels = sentence_dict['ner_tags']
        preds_list.append([id2label[p] for p in preds])
        labels_list.append([id2label[l] for l in labels])

baseline_results = seqeval.compute(predictions=preds_list, references=labels_list)
baseline_f1 = baseline_results['overall_f1']
print(f"Baseline Val F1: {baseline_f1:.4f}")

Baseline Val F1: 0.7815


## 3.4 My method of text processing

In [ ]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT, num_labels=len(label_list), id2label=id2label, label2id=label2id
)

training_args = TrainingArguments(
    output_dir="./roberta-results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=20,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    bf16=True,
    logging_dir='./logs',
    logging_steps=50,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)

Some weights of RobertaForTokenClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_875/2803768789.py:42: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:

print("Training SOTA RoBERTa...")
trainer.train()
trainer.save_model("models/best_model")
print("Best model saved.")

Training SOTA RoBERTa...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.204600,0.185794,0.776596,0.905226,0.835992,0.938560
2,0.145800,0.156071,0.823767,0.902569,0.861369,0.947785
3,0.105400,0.160293,0.844426,0.899026,0.870871,0.950914
4,0.072400,0.169135,0.847500,0.900797,0.873336,0.952397
5,0.051800,0.184812,0.837971,0.906997,0.871119,0.951408
6,0.035900,0.199786,0.861277,0.896368,0.878472,0.956020
7,0.028800,0.208672,0.853904,0.900797,0.876724,0.955362
8,0.022900,0.213303,0.862010,0.896368,0.878854,0.956844


Best model saved.


In [ ]:
history = trainer.state.log_history
train_loss = [x['loss'] for x in history if 'loss' in x]
train_steps = [x['step'] for x in history if 'loss' in x]
val_f1 = [x['eval_f1'] for x in history if 'eval_f1' in x]
val_epochs = [x['epoch'] for x in history if 'eval_f1' in x]

plt.figure(figsize=(12, 6))
plt.plot(train_steps, train_loss, label='RoBERTa Train Loss', color='#1f77b4', linewidth=2)
plt.title('Training Loss over Steps', fontsize=16)
plt.xlabel('Training Steps', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12, loc='upper right')
plt.tight_layout()
plt.savefig('./plots/training_loss.png', dpi=300)
plt.close()

plt.figure(figsize=(12, 6))
plt.plot(val_epochs, val_f1, label='RoBERTa Val F1', color='#2ca02c', marker='o', linewidth=2)
plt.axhline(y=baseline_f1, color='#d62728', linestyle='--', linewidth=2, label=f"Baseline LSTM F1 ({baseline_f1:.4f})")
plt.title('Validation F1 Score Comparison', fontsize=16)
plt.xlabel('Epochs', fontsize=12)
plt.ylabel('F1 Score', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(fontsize=12, loc='lower right')
plt.tight_layout()
plt.savefig('./plots/validation_f1.png', dpi=300)
plt.close()

sota_eval = trainer.evaluate()
df_compare = pd.DataFrame({
    "Model": ["Baseline LSTM", "RoBERTa (Best)"],
    "Val F1": [baseline_f1, sota_eval['eval_f1']],
    "Val Accuracy": [baseline_results['overall_accuracy'], sota_eval['eval_accuracy']]
})
df_compare.to_csv("./tables/model_comparison.csv", index=False)
print("Analysis complete. Files saved to plots/ and tables/.")

Analysis complete. Files saved to plots/ and tables/.


In [ ]:
def generate_submission(model, hf_dataset, raw_dataset, output_name):
    original_model = trainer.model
    trainer.model = model
    
    print(f"Predicting for {output_name}...")
    predictions, _, _ = trainer.predict(hf_dataset)
    predictions = np.argmax(predictions, axis=2)
    final_preds = []
    
    for i, prediction in enumerate(predictions):
        tokens = hf_dataset[i]["tokens"]
        encoding = tokenizer(tokens, is_split_into_words=True, truncation=True)
        word_ids = encoding.word_ids()
        
        word_level_pred = []
        previous_word_idx = None
        
        for idx, word_idx in enumerate(word_ids):
            if word_idx is None: continue
            if word_idx != previous_word_idx:
                word_level_pred.append(id2label[prediction[idx]])
                previous_word_idx = word_idx
        final_preds.append(word_level_pred)
    
    tsv_path = f"submissions/{output_name}.tsv"
    with open(tsv_path, "w", encoding='utf-8') as w:
        for original, pred_tags in zip(raw_dataset, final_preds):
            words = original["tokens"]
            count = min(len(words), len(pred_tags))
            if len(pred_tags) < len(words):
                pred_tags += ["O"] * (len(words) - len(pred_tags))
            
            for k in range(count):
                w.write(f"{words[k]}\t{pred_tags[k]}\n")
            w.write("\n")
            
    zip_path = f"submissions/{output_name}.zip"
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
        zipf.write(tsv_path, arcname="out_test.tsv")
        
    print(f"Saved {tsv_path} and {zip_path}")
    trainer.model = original_model 

best_model = trainer.model
generate_submission(best_model, tokenized_datasets["dev"], raw_dev_data, "dev_best")
generate_submission(best_model, tokenized_datasets["test"], raw_test_data, "test_best")

checkpoints = [d for d in os.listdir("./roberta-results") if d.startswith("checkpoint")]
checkpoints.sort(key=lambda x: int(x.split("-")[1]))
last_ckpt_path = os.path.join("./roberta-results", checkpoints[-1])

print(f"Loading last model from {last_ckpt_path}...")
last_model = AutoModelForTokenClassification.from_pretrained(last_ckpt_path).to(device)

generate_submission(last_model, tokenized_datasets["dev"], raw_dev_data, "dev_last")
generate_submission(last_model, tokenized_datasets["test"], raw_test_data, "test_last")

Predicting for dev_best...


Saved submissions/dev_best.tsv and submissions/dev_best.zip
Predicting for test_best...


Saved submissions/test_best.tsv and submissions/test_best.zip
Loading last model from ./roberta-results/checkpoint-1056...
Predicting for dev_last...


Saved submissions/dev_last.tsv and submissions/dev_last.zip
Predicting for test_last...


Saved submissions/test_last.tsv and submissions/test_last.zip
